In [ ]:
import pandas as pd
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import torch.nn.functional as F
from pathlib import Path
import torch.nn as nn


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

resnet = models.resnet18(weights=None)
resnet.fc = nn.Linear(resnet.fc.in_features, 200)


FINETUNED_MODEL_PATH = "resnet18_tinyimagenet_fc.pth" 
resnet.load_state_dict(torch.load(FINETUNED_MODEL_PATH, map_location=device))
resnet.to(device)

resnet.eval()

transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])


import os

TRAIN_DIR = "ImageNet-ES-Diverse/ImageNet-ES-Diverse/es-train/tin_no_resize_sample_removed"

train_classes = sorted(os.listdir(TRAIN_DIR))
wnid_to_idx = {cls_name: i for i, cls_name in enumerate(train_classes)}
print(f"Locked mapping dictionary with {len(wnid_to_idx)} target classes.")



print(f"Loaded {len(wnid_to_idx)} ImageNet classes.")


df = pd.read_csv("data\oracle_test_200 .csv")

df = df.iloc[:600].reset_index(drop=True)
print(f"Evaluating on {len(df)} test samples.")


top1_correct = 0
top5_correct = 0
total = len(df)

for i, row in df.iterrows():

    csv_path = Path(row["image_path"])

    class_name = csv_path.parent.name
    filename = csv_path.name
    lighting = row["lighting"] 
    base_dir = csv_path.parents[4]
    
    
    image_path = os.path.join('ImageNet-ES-Diverse/', base_dir, "auto_exposure", lighting, "param_1", class_name, filename)
    
   
    if not os.path.exists(image_path):
        continue


    image = Image.open(image_path).convert("RGB")
    image = transform(image).unsqueeze(0).to(device)

    
    class_name = Path(image_path).parent.name

    if class_name not in wnid_to_idx:
        print("not")
        continue

    target = row["label"]
   

    with torch.no_grad():

        output = resnet(image)

       
        prediction = output.argmax(dim=1).item()

        if prediction == target:
            top1_correct += 1

       
        top5 = torch.topk(output, k=5, dim=1).indices.squeeze(0)

        if target in top5:
            top5_correct += 1

    if (i + 1) % 50 == 0:
        print(f"Processed {i+1}/{total} images")


top1_acc = 100 * top1_correct / total
top5_acc = 100 * top5_correct / total

print("\n===============================")
print("Baseline 1 : Auto Exposure")
print("===============================")
print(f"Images Evaluated : {total}")
print(f"Top-1 Accuracy   : {top1_acc:.2f}%")
print(f"Top-5 Accuracy   : {top5_acc:.2f}%")

In [ ]:
import json
import urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import torch.nn as nn


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


resnet = models.resnet18(weights=None)
resnet.fc = nn.Linear(resnet.fc.in_features, 200)

FINETUNED_MODEL_PATH = "resnet18_tinyimagenet_fc.pth" 
resnet.load_state_dict(torch.load(FINETUNED_MODEL_PATH, map_location=device))
resnet.to(device)

resnet.eval()


transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])


import os

TRAIN_DIR = "ImageNet-ES-Diverse/ImageNet-ES-Diverse/es-train/tin_no_resize_sample_removed"

train_classes = sorted(os.listdir(TRAIN_DIR))
wnid_to_idx = {cls_name: i for i, cls_name in enumerate(train_classes)}
print(f"Locked mapping dictionary with {len(wnid_to_idx)} target classes.")

print(f"Loaded {len(wnid_to_idx)} ImageNet classes.")

df = pd.read_csv("data\oracle_test_200 .csv")

df = df.iloc[:600].reset_index(drop=True)
print(f"Evaluating on {len(df)} test samples.")

# ============================================================
# EVALUATION
# ============================================================
top1_correct = 0
top5_correct = 0

total = len(df)

for i, row in df.iterrows():

    csv_path = Path(row["image_path"])
    
    
    class_name = csv_path.parent.name
    filename = csv_path.name
    lighting = row["lighting"] 
    
   
    base_dir = csv_path.parents[4]
    
   
    image_path = os.path.join('ImageNet-ES-Diverse/', base_dir, "auto_exposure", lighting, "param_1", class_name, filename)

    
    confidence_vector = np.fromstring(row['all_27_confidences'].strip("[]"), sep=",")
    oracle_param = int(torch.argmax(torch.tensor(confidence_vector))) + 1
    

    light_env = row["lighting"]
    ae_path = Path(image_path)

    class_name = ae_path.parent.name
    filename = ae_path.name

    
    oracle_path = (
      ae_path.parents[4]
      / "param_control"
      / light_env
      / f"param_{oracle_param}"
      / class_name
      / filename
  )


    if not oracle_path.exists():
        continue

    image = Image.open(oracle_path).convert("RGB")
    image = transform(image).unsqueeze(0).to(device)

    target = wnid_to_idx[class_name]

    with torch.no_grad():

        output = resnet(image)
        output = output 

        
        prediction = output.argmax(dim=1).item()

        if prediction == target:
            top1_correct += 1

        top5 = torch.topk(output, 5, dim=1).indices.squeeze(0)

        if target in top5:
            top5_correct += 1

    if (i + 1) % 50 == 0:
        print(f"Processed {i+1}/{total}")

top1_acc = 100 * top1_correct / total
top5_acc = 100 * top5_correct / total

print("\n===============================")
print("Baseline 2 : Oracle Parameters")
print("===============================")
print(f"Images Evaluated : {total}")
print(f"Top-1 Accuracy   : {top1_acc:.2f}%")
print(f"Top-5 Accuracy   : {top5_acc:.2f}%")

In [ ]:
import json
import urllib.request
from pathlib import Path

import pandas as pd
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


_resnet_backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT).to(device)
feature_extractor = nn.Sequential(*list(_resnet_backbone.children())[:-1]).to(device)
feature_extractor.eval()

for param in feature_extractor.parameters():
    param.requires_grad = False


def extract_features(image_tensor):
   
    r, g, b = image_tensor[0], image_tensor[1], image_tensor[2]
    _, H, W = image_tensor.shape
    features = []

  
    grid = 2
    h_step, w_step = H // grid, W // grid
    for row in range(grid):
        for col in range(grid):
            ys, ye = row * h_step, (row + 1) * h_step
            xs, xe = col * w_step, (col + 1) * w_step
            n_pixels = h_step * w_step
            for channel in [r, g, b]:
                block = channel[ys:ye, xs:xe]
                hist = torch.histc(block, bins=5, min=0.0, max=1.0)
                features.append(hist / n_pixels)

    for channel in [r, g, b]:
        ch_mean = channel.mean()
        ch_std = channel.std()
        clipping = ((channel > 0.95) | (channel < 0.05)).float().mean()
        features.append(torch.stack([ch_mean, ch_std, clipping]))

    gray = 0.2989 * r + 0.5870 * g + 0.1140 * b
    block_means = []
    for row in range(grid):
        for col in range(grid):
            ys, ye = row * h_step, (row + 1) * h_step
            xs, xe = col * w_step, (col + 1) * w_step
            block_means.append(gray[ys:ye, xs:xe].mean())
    block_means_t = torch.stack(block_means)
    global_contrast = block_means_t.std()
    features.append(block_means_t)
    features.append(global_contrast.unsqueeze(0))

    with torch.no_grad():
        image_batch = image_tensor.unsqueeze(0).to(device)
        deep_features = feature_extractor(image_batch)
        deep_features = deep_features.flatten()

    handcrafted = torch.cat([f.flatten() for f in features])
    final = torch.cat([handcrafted, deep_features.cpu()])

    return final



class LensPredictorNetwork(nn.Module):

    def __init__(self, input_dim):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, 27)
        )

    def forward(self, x):
        return self.network(x)



INPUT_DIM = 586
MODEL_PATH = "output\resnet_histogram_3layer_v3.pth"   

lens_model = LensPredictorNetwork(INPUT_DIM).to(device)

lens_model.load_state_dict(
    torch.load(MODEL_PATH, map_location=device)
)

lens_model.eval()

print("Lens predictor loaded.")


resnet = models.resnet18(weights=None)
resnet.fc = nn.Linear(resnet.fc.in_features, 200)


FINETUNED_MODEL_PATH = "resnet18_tinyimagenet_fc.pth" 
resnet.load_state_dict(torch.load(FINETUNED_MODEL_PATH, map_location=device))
resnet.to(device)

resnet.eval()

print("ImageNet ResNet loaded.")



transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])



import os

TRAIN_DIR = "ImageNet-ES-Diverse/ImageNet-ES-Diverse/es-train/tin_no_resize_sample_removed"

train_classes = sorted(os.listdir(TRAIN_DIR))
wnid_to_idx = {cls_name: i for i, cls_name in enumerate(train_classes)}
print(f"Fine tuned model with {len(wnid_to_idx)} target classes.")

print(f"Loaded {len(wnid_to_idx)} ImageNet classes.")



df = pd.read_csv("oracle_test_200 .csv")
df = df.iloc[:600].reset_index(drop=True)

print(f"Evaluating on {len(df)} test samples.")


top1_correct = 0
top5_correct = 0
total_valid = 0

print("Evaluating Your Trained LensPredictor Pipeline...")

for i, row in df.iterrows():
    csv_path = Path(row["image_path"])
    
    # Extract metadata
    class_name = csv_path.parent.name
    filename = csv_path.name
    lighting = row["lighting"] 
    target = int(row["label"])
    
   
    base_dir = csv_path.parents[4]
    
   
    ae_path = Path(os.path.join('ImageNet-ES-Diverse/', base_dir, "auto_exposure", lighting, "param_1", class_name, filename))
    
    if not ae_path.exists():
        continue

    
    ae_image = Image.open(ae_path).convert("RGB")
    ae_tensor = transform(ae_image)

    feat = extract_features(ae_tensor).unsqueeze(0).to(device)
    with torch.no_grad():
        output = lens_model(feat)
        predicted_param = output.argmax(dim=1).item() +1

   
    param_path = Path(os.path.join('ImageNet-ES-Diverse/', base_dir, "param_control", lighting, f"param_{predicted_param}", class_name, filename))

    if not param_path.exists():
        continue 

   
    img = Image.open(param_path).convert("RGB")
    img_tensor = transform(img).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = resnet(img_tensor)
        
        pred = logits.argmax(dim=1).item()
        
        if pred == target:
            top1_correct += 1

        top5 = torch.topk(logits, k=5, dim=1).indices.squeeze(0)
        if target in top5:
            top5_correct += 1

   
    total_valid += 1
    if total_valid % 50 == 0:
        print(f"Processed {total_valid} valid images...")


print("\n===============================")
print("LensPredictor Pipeline")
print("===============================")
if total_valid == 0:
    print("ERROR: 0 images evaluated. Check if your param_path exists on your hard drive!")
else:
    print(f"Images Evaluated : {total_valid}")
    print(f"Top-1 Accuracy   : {100 * top1_correct / total_valid:.2f}%")
    print(f"Top-5 Accuracy   : {100 * top5_correct / total_valid:.2f}%")